# 🚰 Predicción de Potabilidad del Agua con Machine Learning
## Módulo de Clasificación - Búsqueda del Mejor Modelo

**Objetivo:** Construir un modelo de clasificación binaria que determine si una muestra de agua es potable (`1`) o no (`0`) basándose en 9 parámetros químicos y físicos.

**Metodología:**
1. Análisis Exploratorio de Datos (EDA)
2. Preprocesamiento robusto (IterativeImputer MICE + StandardScaler)
3. Análisis de multicolinealidad (VIF)
4. Entrenamiento de 5 modelos con GridSearchCV + Validación Cruzada (K=5)
5. Selección del mejor modelo por ROC AUC
6. Análisis de umbrales óptimos (Youden, Riesgo, Financiero)
7. Empaquetado del modelo con joblib
8. Despliegue en Dash de Streamlit

In [ ]:
# =========================================================================
# IMPORTACIÓN DE LIBRERÍAS
# =========================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Librerías cargadas correctamente")

## 1. Carga y Exploración de Datos (EDA)
Cargamos el dataset y realizamos un reconocimiento inicial de su estructura, tipos de datos y dimensiones.

In [ ]:
# =========================================================================
# CARGA DEL DATASET
# =========================================================================
df = pd.read_csv('D:\\wpardo\\Downloads\\water_potability_project\\data\\water_potability.csv')

print("="*65)
print("=== DATASET: POTABILIDAD DEL AGUA ===")
print("="*65)
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print("\n📋 Primeras 5 filas:")
display(df.head())

print("\n📊 Información general:")
df.info()

## 2. Análisis de Valores Nulos
Detectamos y cuantificamos los valores faltantes. Este dataset tiene nulos en `ph`, `Sulfate` y `Trihalomethanes`, lo que justifica el uso de **IterativeImputer (MICE)** en lugar de una imputación simple.

In [ ]:
# =========================================================================
# ANÁLISIS DE VALORES NULOS
# =========================================================================
missing_values = df.isna().sum()
total_rows = df.shape[0]
missing_vars = missing_values[missing_values > 0]

missing_info_df = pd.DataFrame({
    'Datos faltantes': missing_vars,
    'Porcentaje (%)': round((missing_vars / total_rows) * 100, 2)
})

print("="*65)
print("=== ANÁLISIS DE VALORES NULOS ===")
print("="*65)
display(missing_info_df)

# Heatmap de valores nulos
plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Mapa de Calor de Valores Nulos', fontsize=14, weight='bold')
plt.xlabel('Características')
plt.ylabel('Muestras')
plt.tight_layout()
plt.show()

## 3. Distribución de la Variable Objetivo
Analizamos el balance de clases. Un dataset desbalanceado requiere métricas como **ROC AUC** y **F1-Score** en lugar de solo Accuracy.

In [ ]:
# =========================================================================
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# =========================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conteo absoluto
sns.countplot(x='Potability', data=df, palette='viridis', ax=axes[0])
axes[0].set_title('Distribución Absoluta', fontsize=14, weight='bold')
axes[0].set_xlabel('Potability (0: No Potable, 1: Potable)')
axes[0].set_ylabel('Frecuencia')

# Proporción
target_counts = df['Potability'].value_counts(normalize=True) * 100
axes[1].pie(target_counts, labels=['No Potable (0)', 'Potable (1)'], 
            autopct='%1.1f%%', colors=sns.color_palette('viridis'), startangle=90)
axes[1].set_title('Proporción de Clases (%)', fontsize=14, weight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Distribución de clases:")
print(df['Potability'].value_counts())
print(f"\nProporción: {df['Potability'].value_counts(normalize=True).to_dict()}")

## 4. Análisis Descriptivo por Clase
Visualizamos cómo se distribuye cada variable química según la potabilidad del agua usando boxplots.

In [ ]:
# =========================================================================
# ANÁLISIS DESCRIPTIVO POR CLASE
# =========================================================================
variables_numericas = df.select_dtypes(include=[np.number]).columns.drop('Potability')

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(variables_numericas):
    sns.boxplot(x='Potability', y=col, data=df, ax=axes[i], palette='viridis')
    axes[i].set_title(f'{col} vs Potability', fontsize=12, weight='bold')
    axes[i].set_xlabel('Potability')

plt.tight_layout()
plt.suptitle('Distribución de Variables Químicas según Potabilidad', 
             fontsize=16, weight='bold', y=1.02)
plt.show()

## 5. Matriz de Correlación
Identificamos relaciones lineales entre variables. Valores cercanos a ±1 indican alta correlación (posible multicolinealidad).

In [ ]:
# =========================================================================
# MATRIZ DE CORRELACIÓN
# =========================================================================
plt.figure(figsize=(12, 9))
correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Matriz de Correlación entre Variables', fontsize=14, weight='bold', pad=20)
plt.tight_layout()
plt.show()

## 6. Separación de Features y Target
Separamos las variables predictoras (X) de la variable objetivo (y).

In [ ]:
# =========================================================================
# SEPARACIÓN DE FEATURES Y TARGET
# =========================================================================
X = df.drop(columns=['Potability'])
y = df['Potability']

col_numericas = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"✅ Variables predictoras: {len(col_numericas)}")
print(f"📋 Columnas: {col_numericas}")
print(f"🎯 Target shape: {y.shape}")

## 7. División Estratificada (80/20)
Usamos `stratify=y` para mantener la proporción de clases en ambos conjuntos. Esto evita sesgos en la evaluación.

In [ ]:
from sklearn.model_selection import train_test_split

# =========================================================================
# DIVISIÓN ESTRATIFICADA
# =========================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    stratify=y,
    random_state=42
)

print("- VERIFICACIÓN DEL SPLIT ESTRATIFICADO -")
print(f"Tamaño Train (X_train): {X_train.shape}")
print(f"Tamaño Test (X_test): {X_test.shape}")
print(f"\nDistribución en Train:\n{y_train.value_counts(normalize=True)}")
print(f"\nDistribución en Test:\n{y_test.value_counts(normalize=True)}")

## 8. Imputación Iterativa (MICE)
En lugar de rellenar con la media (que destruye relaciones entre variables), usamos **IterativeImputer** que modela cada variable con nulos en función de las demás. Es la técnica más robusta del módulo.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# =========================================================================
# IMPUTACIÓN ITERATIVA (MICE)
# =========================================================================
X_train_imp = X_train.copy()
X_test_imp = X_test.copy()

iterative_imp = IterativeImputer(random_state=42, max_iter=10)

X_train_imp[col_numericas] = iterative_imp.fit_transform(X_train[col_numericas])
X_test_imp[col_numericas] = iterative_imp.transform(X_test[col_numericas])

print("- CONTROL DE CALIDAD (NULOS DESPUÉS DE IMPUTACIÓN) -")
print(f"Nulos en X_train: {X_train_imp.isna().sum().sum()}")
print(f"Nulos en X_test: {X_test_imp.isna().sum().sum()}")

## 9. Análisis de Multicolinealidad (VIF)
El **Factor de Inflación de la Varianza (VIF)** mide cuánto se infla la varianza de un coeficiente debido a correlaciones con otras variables. 
- VIF < 5: Aceptable ✅
- VIF 5-10: Preocupante ⚠️
- VIF > 10: Multicolinealidad severa ❌

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# =========================================================================
# ANÁLISIS DE MULTICOLINEALIDAD (VIF)
# =========================================================================
X_vif = X_train_imp.copy()
X_vif['intercept'] = 1

vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) 
                   for i in range(X_vif.shape[1])]

vif_data = vif_data[vif_data["Variable"] != 'intercept'].sort_values(
    by="VIF", ascending=False)

print("=== REPORTE DE MULTICOLINEALIDAD GLOBAL (VIF) ===")
print(vif_data.to_string(index=False))
print("\n💡 Nota: VIF < 5 indica multicolinealidad aceptable")

## 10. Estandarización de Variables
Aplicamos `StandardScaler` para convertir todas las variables a media=0 y varianza=1. Es **obligatorio** para modelos basados en distancias (KNN, SVM) y mejora la convergencia de Regresión Logística.

In [ ]:
from sklearn.preprocessing import StandardScaler

# =========================================================================
# ESCALADO ESTÁNDAR
# =========================================================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=col_numericas, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=col_numericas, index=X_test.index)

print("- CONTROL DE CALIDAD (DATA ESCALADA) -")
print(f"Dimensiones X_train_scaled: {X_train_scaled_df.shape}")
print(f"Dimensiones X_test_scaled: {X_test_scaled_df.shape}")
display(X_train_scaled_df.describe())

## 11. Función Auxiliar de Evaluación
Creamos una función reutilizable que calcula métricas clave: **ROC AUC, Accuracy, Recall, F1-Score** y muestra la matriz de confusión.

In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve
)

# =========================================================================
# FUNCIÓN AUXILIAR PARA EVALUAR MODELOS
# =========================================================================
def evaluar_modelo(nombre_modelo, mejor_pipeline, X_train, X_test, y_train, y_test):
    """Evalúa un modelo en train y test con múltiples métricas"""
    
    y_pred_train = mejor_pipeline.predict(X_train)
    y_pred_proba_train = mejor_pipeline.predict_proba(X_train)[:, 1]
    y_pred_test = mejor_pipeline.predict(X_test)
    y_pred_proba_test = mejor_pipeline.predict_proba(X_test)[:, 1]
    
    roc_auc_train = roc_auc_score(y_train, y_pred_proba_train)
    accuracy_train = accuracy_score(y_train, y_pred_train)
    recall_train = recall_score(y_train, y_pred_train)
    f1_train = f1_score(y_train, y_pred_train)
    
    roc_auc_test = roc_auc_score(y_test, y_pred_proba_test)
    accuracy_test = accuracy_score(y_test, y_pred_test)
    recall_test = recall_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test)
    
    print("\n" + "="*65)
    print(f"=== REPORTE COMPLETO: {nombre_modelo} ===")
    print("="*65)
    
    print(f"\n📈 RENDIMIENTO EN ENTRENAMIENTO (TRAIN)")
    print("-"*65)
    print(f"ROC AUC (Train): {roc_auc_train:.4f}")
    print(f"Accuracy (Train): {accuracy_train:.4f}")
    print(f"Recall (Train): {recall_train:.4f}")
    print(f"F1-Score (Train): {f1_train:.4f}")
    
    print(f"\n📉 RENDIMIENTO EN PRUEBA (TEST)")
    print("-"*65)
    print(f"ROC AUC (Test): {roc_auc_test:.4f}")
    print(f"Accuracy (Test): {accuracy_test:.4f}")
    print(f"Recall (Test): {recall_test:.4f}")
    print(f"F1-Score (Test): {f1_test:.4f}")
    
    print(f"\n📋 MATRIZ DE CONFUSIÓN (TEST)")
    print("-"*65)
    print(confusion_matrix(y_test, y_pred_test))
    
    print(f"\n📊 REPORTE DE CLASIFICACIÓN (TEST)")
    print("-"*65)
    print(classification_report(y_test, y_pred_test, 
                                target_names=['No Potable (0)', 'Potable (1)']))
    
    return {
        'modelo': nombre_modelo,
        'roc_auc_test': roc_auc_test,
        'accuracy_test': accuracy_test,
        'recall_test': recall_test,
        'f1_test': f1_test,
        'y_pred_proba_test': y_pred_proba_test,
        'y_pred_test': y_pred_test
    }

## 12. Modelo 1: Regresión Logística
Modelo lineal clásico. Probamos diferentes valores de regularización `C` y penalizaciones.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline

# =========================================================================
# PIPELINE + GRIDSEARCH: REGRESIÓN LOGÍSTICA
# =========================================================================
pipeline_lr = make_pipeline(
    IterativeImputer(random_state=42, max_iter=10),
    StandardScaler(),
    LogisticRegression(random_state=42, max_iter=2000)
)

param_grid_lr = {
    'logisticregression__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'logisticregression__penalty': ['l2'],
    'logisticregression__solver': ['lbfgs', 'saga'],
    'logisticregression__class_weight': [None, 'balanced']
}

print("⏳ Ejecutando GridSearchCV para Regresión Logística...")
grid_search_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_lr.fit(X_train, y_train)

mejor_pipeline_lr = grid_search_lr.best_estimator_
indice_ganador_lr = grid_search_lr.best_index_
desv_estandar_cv_lr = grid_search_lr.cv_results_['std_test_score'][indice_ganador_lr]

print("\n" + "="*65)
print("=== RESULTADOS REGRESIÓN LOGÍSTICA ===")
print("="*65)
print(f"Mejores Hiperparámetros: {grid_search_lr.best_params_}")
print(f"ROC AUC Promedio (CV): {grid_search_lr.best_score_:.4f}")
print(f"Desviación ROC AUC (CV): ±{desv_estandar_cv_lr:.4f}")

resultado_lr = evaluar_modelo("Regresión Logística", mejor_pipeline_lr, 
                               X_train, X_test, y_train, y_test)

## 13. Modelo 2: K-Nearest Neighbors (KNN)
Modelo basado en distancias. Buscamos el número óptimo de vecinos y la métrica de distancia.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# =========================================================================
# PIPELINE + GRIDSEARCH: KNN
# =========================================================================
pipeline_knn = make_pipeline(
    IterativeImputer(random_state=42, max_iter=10),
    StandardScaler(),
    KNeighborsClassifier()
)

param_grid_knn = {
    'kneighborsclassifier__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'kneighborsclassifier__weights': ['uniform', 'distance'],
    'kneighborsclassifier__metric': ['euclidean', 'manhattan', 'minkowski'],
    'kneighborsclassifier__leaf_size': [10, 20, 30, 50]
}

print("⏳ Ejecutando GridSearchCV para KNN...")
grid_search_knn = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=param_grid_knn,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_knn.fit(X_train, y_train)

mejor_pipeline_knn = grid_search_knn.best_estimator_
indice_ganador_knn = grid_search_knn.best_index_
desv_estandar_cv_knn = grid_search_knn.cv_results_['std_test_score'][indice_ganador_knn]

print("\n" + "="*65)
print("=== RESULTADOS KNN ===")
print("="*65)
print(f"Mejores Hiperparámetros: {grid_search_knn.best_params_}")
print(f"ROC AUC Promedio (CV): {grid_search_knn.best_score_:.4f}")
print(f"Desviación ROC AUC (CV): ±{desv_estandar_cv_knn:.4f}")

resultado_knn = evaluar_modelo("KNN", mejor_pipeline_knn, 
                                X_train, X_test, y_train, y_test)

## 14. Modelo 3: SVM Lineal
Support Vector Machine con kernel lineal. Busca el hiperplano óptimo que separa las clases.


In [ ]:
from sklearn.svm import LinearSVC

# =========================================================================
# PIPELINE + GRIDSEARCH: SVM LINEAL
# =========================================================================
pipeline_svc = make_pipeline(
    IterativeImputer(random_state=42, max_iter=10),
    StandardScaler(),
    LinearSVC(random_state=42, max_iter=5000)
)

param_grid_svc = {
    'linearsvc__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'linearsvc__class_weight': [None, 'balanced']
}

print("⏳ Ejecutando GridSearchCV para SVM Lineal...")
grid_search_svc = GridSearchCV(
    estimator=pipeline_svc,
    param_grid=param_grid_svc,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_svc.fit(X_train, y_train)

mejor_pipeline_svc = grid_search_svc.best_estimator_
indice_ganador_svc = grid_search_svc.best_index_
desv_estandar_cv_svc = grid_search_svc.cv_results_['std_test_score'][indice_ganador_svc]

print("\n" + "="*65)
print("=== RESULTADOS SVM LINEAL ===")
print("="*65)
print(f"Mejores Hiperparámetros: {grid_search_svc.best_params_}")
print(f"ROC AUC Promedio (CV): {grid_search_svc.best_score_:.4f}")
print(f"Desviación ROC AUC (CV): ±{desv_estandar_cv_svc:.4f}")

resultado_svc = evaluar_modelo("SVM Lineal", mejor_pipeline_svc, 
                                X_train, X_test, y_train, y_test)

## 15. Modelo 4: Random Forest ⭐
Ensemble de árboles de decisión. Captura relaciones no lineales y es robusto ante outliers.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# =========================================================================
# PIPELINE + GRIDSEARCH: RANDOM FOREST
# =========================================================================
pipeline_rf = make_pipeline(
    IterativeImputer(random_state=42, max_iter=10),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid_rf = {
    'randomforestclassifier__n_estimators': [100, 200, 300, 500],
    'randomforestclassifier__max_depth': [None, 5, 10, 20],
    'randomforestclassifier__min_samples_split': [2, 5, 10],
    'randomforestclassifier__min_samples_leaf': [1, 2, 4],
    'randomforestclassifier__max_features': ['sqrt', 'log2'],
    'randomforestclassifier__bootstrap': [True, False],
    'randomforestclassifier__class_weight': [None, 'balanced', 'balanced_subsample']
}

print("⏳ Ejecutando GridSearchCV para Random Forest...")
grid_search_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train, y_train)

mejor_pipeline_rf = grid_search_rf.best_estimator_
indice_ganador_rf = grid_search_rf.best_index_
desv_estandar_cv_rf = grid_search_rf.cv_results_['std_test_score'][indice_ganador_rf]

print("\n" + "="*65)
print("=== RESULTADOS RANDOM FOREST ===")
print("="*65)
print(f"Mejores Hiperparámetros: {grid_search_rf.best_params_}")
print(f"ROC AUC Promedio (CV): {grid_search_rf.best_score_:.4f}")
print(f"Desviación ROC AUC (CV): ±{desv_estandar_cv_rf:.4f}")

resultado_rf = evaluar_modelo("Random Forest", mejor_pipeline_rf, 
                               X_train, X_test, y_train, y_test)

## 16. Modelo 5: XGBoost ⭐⭐
Gradient Boosting optimizado. Suele ser el **mejor modelo** en datasets tabulares. Incluye `scale_pos_weight` para manejar el desbalance de clases.

In [ ]:
from xgboost import XGBClassifier

# =========================================================================
# PIPELINE + GRIDSEARCH: XGBOOST
# =========================================================================
pipeline_xgb = make_pipeline(
    IterativeImputer(random_state=42, max_iter=10),
    StandardScaler(),
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
)

param_grid_xgb = {
    'xgbclassifier__n_estimators': [100, 200, 300, 500],
    'xgbclassifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'xgbclassifier__max_depth': [3, 5, 7, 10],
    'xgbclassifier__subsample': [0.7, 0.8, 0.9, 1.0],
    'xgbclassifier__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'xgbclassifier__min_child_weight': [1, 3, 5],
    'xgbclassifier__scale_pos_weight': [1, (y_train==0).sum()/(y_train==1).sum()]
}

print("⏳ Ejecutando GridSearchCV para XGBoost...")
grid_search_xgb = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid_xgb,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_xgb.fit(X_train, y_train)

mejor_pipeline_xgb = grid_search_xgb.best_estimator_
indice_ganador_xgb = grid_search_xgb.best_index_
desv_estandar_cv_xgb = grid_search_xgb.cv_results_['std_test_score'][indice_ganador_xgb]

print("\n" + "="*65)
print("=== RESULTADOS XGBOOST ===")
print("="*65)
print(f"Mejores Hiperparámetros: {grid_search_xgb.best_params_}")
print(f"ROC AUC Promedio (CV): {grid_search_xgb.best_score_:.4f}")
print(f"Desviación ROC AUC (CV): ±{desv_estandar_cv_xgb:.4f}")

resultado_xgb = evaluar_modelo("XGBoost", mejor_pipeline_xgb, 
                                X_train, X_test, y_train, y_test)

## 17. Comparación de Modelos
Visualizamos la estabilidad de cada modelo mediante un boxplot del ROC AUC en los 5 folds de validación cruzada.

In [ ]:
# =========================================================================
# EXTRACCIÓN DE ROC AUC POR FOLD
# =========================================================================
modelos_grid = {
    'Regresión Logística': grid_search_lr,
    'KNN': grid_search_knn,
    'SVM Lineal': grid_search_svc,
    'Random Forest': grid_search_rf,
    'XGBoost': grid_search_xgb
}

data_grafico = {}
for nombre, grid in modelos_grid.items():
    idx = grid.best_index_
    data_grafico[nombre] = [
        grid.cv_results_[f'split{i}_test_score'][idx] for i in range(5)
    ]

df_boxplot = pd.DataFrame(data_grafico)

# =========================================================================
# BOXPLOT COMPARATIVO
# =========================================================================
plt.figure(figsize=(12, 7))
ax = sns.boxplot(data=df_boxplot, palette='viridis', orient='h')

plt.title('Comparación de Estabilidad de Modelos\nDistribución del ROC AUC en Validación Cruzada (K=5)', 
          fontsize=15, pad=15, weight='bold')
plt.xlabel('ROC AUC Score', fontsize=12, labelpad=10)
plt.ylabel('Modelos Evaluados', fontsize=12)
plt.xlim([0.5, 0.75])
plt.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Azar (0.5)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 18. Tabla Comparativa Final
Ordenamos los modelos por ROC AUC para identificar el ganador.

In [ ]:
# =========================================================================
# TABLA COMPARATIVA FINAL
# =========================================================================
resultados_comparacion = [resultado_lr, resultado_knn, resultado_svc, 
                          resultado_rf, resultado_xgb]

df_comparacion = pd.DataFrame([
    {
        'Modelo': r['modelo'],
        'ROC AUC (Test)': f"{r['roc_auc_test']:.4f}",
        'Accuracy (Test)': f"{r['accuracy_test']:.4f}",
        'Recall (Test)': f"{r['recall_test']:.4f}",
        'F1-Score (Test)': f"{r['f1_test']:.4f}"
    }
    for r in resultados_comparacion
])

df_comparacion['ROC AUC (Test)_num'] = df_comparacion['ROC AUC (Test)'].astype(float)
df_comparacion = df_comparacion.sort_values('ROC AUC (Test)_num', ascending=False).drop(
    columns=['ROC AUC (Test)_num'])

print("="*65)
print("=== TABLA COMPARATIVA DE MODELOS (ORDENADA POR ROC AUC) ===")
print("="*65)
display(df_comparacion)

mejor_modelo_nombre = df_comparacion.iloc[0]['Modelo']
print(f"\n🏆 MEJOR MODELO: {mejor_modelo_nombre}")

## 19. Análisis Profundo del Mejor Modelo
Generamos la **Curva ROC** para visualizar el trade-off entre TPR (sensibilidad) y FPR.

In [ ]:
# =========================================================================
# CURVA ROC DEL MEJOR MODELO
# =========================================================================
mejores_scores = {
    'Regresión Logística': grid_search_lr.best_score_,
    'KNN': grid_search_knn.best_score_,
    'SVM Lineal': grid_search_svc.best_score_,
    'Random Forest': grid_search_rf.best_score_,
    'XGBoost': grid_search_xgb.best_score_
}
mejor_modelo_nombre = max(mejores_scores, key=mejores_scores.get)
mejor_pipeline = {
    'Regresión Logística': mejor_pipeline_lr,
    'KNN': mejor_pipeline_knn,
    'SVM Lineal': mejor_pipeline_svc,
    'Random Forest': mejor_pipeline_rf,
    'XGBoost': mejor_pipeline_xgb
}[mejor_modelo_nombre]

print(f"🏆 Analizando en profundidad: {mejor_modelo_nombre}")

y_pred_proba_test = mejor_pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
roc_auc = roc_auc_score(y_test, y_pred_proba_test)

plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, color='darkblue', lw=3, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Azar (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
plt.title(f'Curva ROC - {mejor_modelo_nombre}', fontsize=14, weight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 20. Cálculo de Umbrales Óptimos
El umbral por defecto es 0.5, pero podemos optimizarlo según el contexto de negocio:
- **Youden**: Balance matemático perfecto
- **Riesgo**: Control estricto de falsos positivos
- **Financiero**: Maximiza la utilidad neta

In [ ]:
# =========================================================================
# CÁLCULO DE UMBRALES ÓPTIMOS
# =========================================================================

# MÉTODO 1: Índice de Youden
j_scores = tpr - fpr
best_idx_youden = np.argmax(j_scores)
umbral_youden = thresholds[best_idx_youden]

# MÉTODO 2: Riesgo Estricto (FPR <= 5%)
fpr_limit_idx = np.where(fpr <= 0.05)[0]
umbral_riesgo = thresholds[fpr_limit_idx[-1]] if len(fpr_limit_idx) > 0 else 0.5

# MÉTODO 3: Umbral Financiero
ganancia_vp = 100
perdida_fp = -200
utilidades = []
for u in thresholds:
    preds_temp = (y_pred_proba_test >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds_temp).ravel()
    utilidad_neta = (tp * ganancia_vp) + (fp * perdida_fp)
    utilidades.append(utilidad_neta)

umbral_financiero = thresholds[np.argmax(utilidades)]
max_utilidad = max(utilidades)

print("="*65)
print(f"=== CÁLCULO DE UMBRALES ÓPTIMOS - MODELO: {mejor_modelo_nombre} ===")
print("="*65)
print(f"1. Umbral Estadístico (Youden): {umbral_youden:.4f}")
print(f"2. Umbral de Riesgo Estricto (FPR ≤ 5%): {umbral_riesgo:.4f}")
print(f"3. Umbral Financiero Optimizado: {umbral_financiero:.4f}")
print(f"   (Maximiza utilidad neta: ${max_utilidad:,.2f})")

## 21. Importancia de Variables
Si el mejor modelo es Random Forest o XGBoost, extraemos la importancia relativa de cada característica.

In [ ]:
# =========================================================================
# IMPORTANCIA DE VARIABLES
# =========================================================================
if mejor_modelo_nombre in ['Random Forest', 'XGBoost']:
    step_name = 'randomforestclassifier' if mejor_modelo_nombre == 'Random Forest' else 'xgbclassifier'
    modelo_interno = mejor_pipeline.named_steps[step_name]
    
    importancias = modelo_interno.feature_importances_
    
    df_importancia = pd.DataFrame({
        'Variable': col_numericas,
        'Importancia': importancias
    }).sort_values(by='Importancia', ascending=False)
    
    plt.figure(figsize=(10, 7))
    sns.barplot(x='Importancia', y='Variable', data=df_importancia, palette='viridis')
    plt.title(f'Importancia de Variables - {mejor_modelo_nombre}', 
              fontsize=14, weight='bold', pad=15)
    plt.xlabel('Importancia Relativa', fontsize=12)
    plt.ylabel('Variables', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Importancia de variables:")
    display(df_importancia)

## 22. Empaquetado del Mejor Modelo
Guardamos el pipeline completo con `joblib` para despliegue en producción.

In [ ]:
import joblib

# =========================================================================
# EMPAQUETAMIENTO DEL MEJOR MODELO
# =========================================================================
nombre_archivo = f'mejor_modelo_{mejor_modelo_nombre.lower().replace(" ", "_")}_water.joblib'

joblib.dump(mejor_pipeline, nombre_archivo, compress=3)

metadata = {
    'mejor_modelo': mejor_modelo_nombre,
    'umbral_youden': umbral_youden,
    'umbral_financiero': umbral_financiero,
    'columnas': col_numericas,
    'roc_auc_test': roc_auc_score(y_test, y_pred_proba_test)
}
joblib.dump(metadata, 'metadata_modelo.joblib')

print("="*65)
print("🎉 ¡MODELO EMPAQUETADO CON ÉXITO!")
print("="*65)
print(f"📦 Archivo del modelo: {nombre_archivo}")
print(f"📋 Archivo de metadata: metadata_modelo.joblib")
print(f"🏆 Mejor modelo: {mejor_modelo_nombre}")
print(f"📊 ROC AUC en Test: {metadata['roc_auc_test']:.4f}")
print("="*65)

## 23. Dash de Streamlit
Crea un archivo llamado `app.py` con el siguiente código y ejecútalo con:
```bash
streamlit run app.py